<a href="https://colab.research.google.com/github/MLDreamer/AIMathematicallyexplained/blob/main/Hormuz_optimization_attempt_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
pip install gurobipy

In [11]:
"""
================================================================================
THE HORMUZ OPTIMISATION PROBLEM
Two-Stage Stochastic MILP · ANZ Fuel Supply Chain · Gurobi
================================================================================

THE BUG YOU KEPT HITTING — explained once, clearly:

    When l is itself a tuple, e.g. l = ('SRC', 'DC_North'),
    writing  x[l, s, t]  in Python creates the key:
        (('SRC', 'DC_North'), 'mild', 1)   ← nested tuple, 3 elements

    But Gurobi's addVars(ALL_LANES, SCENARIOS, periods) stores keys as:
        ('SRC', 'DC_North', 'mild', 1)     ← flat tuple, 4 elements

    These are DIFFERENT keys. Hence KeyError every time.

    THE FIX: unpack l when indexing x, u, v:
        WRONG:  x[l, s, t]
        RIGHT:  x[l[0], l[1], s, t]   OR   x[(*l, s, t)]

    This file uses x[l[0], l[1], s, t] throughout. Zero ambiguity.

================================================================================
"""

import gurobipy as gp
from gurobipy import GRB
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from copy import deepcopy


# ── Colour palette ────────────────────────────────────────────────────────────
C = {
    'mild':     '#5DCAA5',
    'moderate': '#EF9F27',
    'severe':   '#D85A30',
    'stage1':   '#AFA9EC',
    'shortage': '#F09595',
    'det':      '#888780',
}

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': '#fafaf9',
    'axes.facecolor':   '#fafaf9',
})


# ── Sets ──────────────────────────────────────────────────────────────────────

DCs       = ['DC_North', 'DC_Central', 'DC_South']
DEMANDS   = ['D1', 'D2', 'D3', 'D4', 'D5']
SCENARIOS = ['mild', 'moderate', 'severe']
PERIODS   = [1, 2]

supply_lanes = [
    ('SRC', 'DC_North'),
    ('SRC', 'DC_Central'),
    ('SRC', 'DC_South'),
]

dc_demand_lanes = [
    ('DC_North',   'D1'), ('DC_North',   'D2'),
    ('DC_Central', 'D2'), ('DC_Central', 'D3'), ('DC_Central', 'D4'),
    ('DC_South',   'D3'), ('DC_South',   'D4'), ('DC_South',   'D5'),
]

ALL_LANES = supply_lanes + dc_demand_lanes


# ── Parameters ────────────────────────────────────────────────────────────────

BASE_PARAMS = {
    'prob':        {'mild': 0.25, 'moderate': 0.50, 'severe': 0.25},
    'cost_mult':   {'mild': 1.10, 'moderate': 1.30, 'severe': 1.50},
    'demand_mult': {'mild': 1.20, 'moderate': 1.80, 'severe': 3.00},

    # ALL lanes in ALL_LANES must appear here — both supply and demand lanes
    'base_cost': {
        ('SRC', 'DC_North'):   2.0,
        ('SRC', 'DC_Central'): 1.8,
        ('SRC', 'DC_South'):   2.1,
        ('DC_North',   'D1'):  0.6,
        ('DC_North',   'D2'):  0.8,
        ('DC_Central', 'D2'):  0.5,
        ('DC_Central', 'D3'):  0.5,
        ('DC_Central', 'D4'):  0.7,
        ('DC_South',   'D3'):  0.9,
        ('DC_South',   'D4'):  0.6,
        ('DC_South',   'D5'):  0.8,
    },

    'fixed_lane_cost': {
        ('SRC', 'DC_North'):   800,
        ('SRC', 'DC_Central'): 700,
        ('SRC', 'DC_South'):   900,
        **{l: 200 for l in dc_demand_lanes},
    },

    'lane_cap': {
        **{l: 100 for l in supply_lanes},
        **{l:  60 for l in dc_demand_lanes},
    },

    'init_inv':    {'DC_North': 80, 'DC_Central': 90, 'DC_South': 70},
    'base_demand': {'D1': 30, 'D2': 40, 'D3': 35, 'D4': 25, 'D5': 20},
    'M':           50.0,
    'periods':     [1, 2],
}


# ── Validation ────────────────────────────────────────────────────────────────

def validate(params):
    """Every lane in ALL_LANES must have a base_cost. Fail loudly if not."""
    missing = [l for l in ALL_LANES if l not in params['base_cost']]
    if missing:
        raise ValueError(
            f"\nbase_cost is missing entries for these lanes:\n  {missing}\n"
            "Every lane in ALL_LANES must have a cost entry."
        )

validate(BASE_PARAMS)


# ── Build and solve ───────────────────────────────────────────────────────────

def build_and_solve(params, silent=True):
    """
    Build and solve the two-stage stochastic MILP.

    KEY INDEXING RULE throughout this function:
        x, u, v variables are accessed as  x[src, dst, scenario, period]
        where src, dst come from unpacking the lane tuple l = (src, dst).
        NEVER write  x[l, s, t]  when l is itself a tuple.
        ALWAYS write  x[l[0], l[1], s, t].
    """
    validate(params)

    prob_s      = params['prob']
    cost_mult   = params['cost_mult']
    demand_mult = params['demand_mult']
    base_cost   = params['base_cost']
    fixed_lc    = params['fixed_lane_cost']
    lane_cap    = params['lane_cap']
    init_inv    = params['init_inv']
    base_dem    = params['base_demand']
    M           = params['M']
    periods     = params['periods']

    # Precompute lane costs: dict[(src, dst, scenario)] = float
    lane_cost = {
        (l[0], l[1], s): base_cost[l] * cost_mult[s]
        for l in ALL_LANES
        for s in SCENARIOS
    }

    # Demand: dict[(demand_node, scenario, period)] = float
    demand = {
        (d, s, t): base_dem[d] * demand_mult[s] * (1.1 if t == 2 else 1.0)
        for d in DEMANDS
        for s in SCENARIOS
        for t in periods
    }

    mdl = gp.Model('hormuz_stochastic_milp')
    mdl.setParam('OutputFlag', 0 if silent else 1)
    mdl.setParam('MIPGap', 0.001)

    # ── Stage 1 variables ─────────────────────────────────────────────────────
    # y[(src, dst)] — binary, one per supply lane
    y = mdl.addVars(
        [(l[0], l[1]) for l in supply_lanes],
        vtype=GRB.BINARY,
        name='y'
    )

    # b[dc_name] — continuous buffer stock per DC
    b = mdl.addVars(DCs, lb=0.0, name='b')

    # ── Stage 2 variables ─────────────────────────────────────────────────────
    # x[(src, dst, scenario, period)] — flow on each lane
    x = mdl.addVars(
        [(l[0], l[1], s, t)
         for l in ALL_LANES
         for s in SCENARIOS
         for t in periods],
        lb=0.0,
        name='x'
    )

    # u[(demand_node, scenario, period)] — shortage
    u = mdl.addVars(
        [(d, s, t)
         for d in DEMANDS
         for s in SCENARIOS
         for t in periods],
        lb=0.0,
        name='u'
    )

    # v[(dc_name, scenario, period)] — inventory
    v = mdl.addVars(
        [(n, s, t)
         for n in DCs
         for s in SCENARIOS
         for t in periods],
        lb=0.0,
        name='v'
    )

    # ── Objective ─────────────────────────────────────────────────────────────
    # Term 1: Stage 1 fixed lane costs (paid regardless of scenario)
    obj_stage1 = gp.quicksum(
        fixed_lc[l] * y[l[0], l[1]]
        for l in supply_lanes
    )

    # Term 2 + 3: Expected Stage 2 cost (fuel + shortage), weighted by p_s
    obj_stage2 = gp.quicksum(
        prob_s[s] * (
            gp.quicksum(
                lane_cost[l[0], l[1], s] * x[l[0], l[1], s, t]
                for l in ALL_LANES
                for t in periods
            )
            + M * gp.quicksum(
                u[d, s, t]
                for d in DEMANDS
                for t in periods
            )
        )
        for s in SCENARIOS
    )

    mdl.setObjective(obj_stage1 + obj_stage2, GRB.MINIMIZE)

    # ── C1: No flow on a closed supply lane ───────────────────────────────────
    for l in supply_lanes:
        for s in SCENARIOS:
            for t in periods:
                mdl.addConstr(
                    x[l[0], l[1], s, t] <= lane_cap[l] * y[l[0], l[1]],
                    name=f'C1_{l[0]}_{l[1]}_{s}_{t}'
                )

    # ── C2: Lane capacity ceiling ─────────────────────────────────────────────
    for l in ALL_LANES:
        for s in SCENARIOS:
            for t in periods:
                mdl.addConstr(
                    x[l[0], l[1], s, t] <= lane_cap[l],
                    name=f'C2_{l[0]}_{l[1]}_{s}_{t}'
                )

    # ── C3: Flow balance at each DC ───────────────────────────────────────────
    for n in DCs:
        in_lanes  = [l for l in supply_lanes    if l[1] == n]
        out_lanes = [l for l in dc_demand_lanes if l[0] == n]
        for s in SCENARIOS:
            for t in periods:
                prev_inv = (
                    init_inv[n] if t == periods[0]
                    else v[n, s, t - 1]
                )
                mdl.addConstr(
                    v[n, s, t]
                    == prev_inv
                    + gp.quicksum(x[l[0], l[1], s, t] for l in in_lanes)
                    - gp.quicksum(x[l[0], l[1], s, t] for l in out_lanes),
                    name=f'C3_{n}_{s}_{t}'
                )

    # ── C4: Inventory stays at or above buffer floor ──────────────────────────
    for n in DCs:
        for s in SCENARIOS:
            for t in periods:
                mdl.addConstr(
                    v[n, s, t] >= b[n],
                    name=f'C4_{n}_{s}_{t}'
                )

    # ── C5: Demand met by flow or shortage ────────────────────────────────────
    for d in DEMANDS:
        serving = [l for l in dc_demand_lanes if l[1] == d]
        for s in SCENARIOS:
            for t in periods:
                mdl.addConstr(
                    gp.quicksum(x[l[0], l[1], s, t] for l in serving)
                    + u[d, s, t]
                    >= demand[d, s, t],
                    name=f'C5_{d}_{s}_{t}'
                )

    # ── C6: Buffer cannot exceed current inventory ────────────────────────────
    for n in DCs:
        mdl.addConstr(b[n] <= init_inv[n], name=f'C6_{n}')

    # ── Solve ─────────────────────────────────────────────────────────────────
    mdl.optimize()

    if mdl.Status == GRB.OPTIMAL:
        return mdl, y, b, x, u, v
    else:
        print(f'  Status code: {mdl.Status}')
        return None


# ── Extract results ───────────────────────────────────────────────────────────

def extract(result, params):
    mdl, y, b, x, u, v = result
    periods = params['periods']

    r = {
        'obj': mdl.ObjVal,
        'y':   {l: int(round(y[l[0], l[1]].X)) for l in supply_lanes},
        'b':   {n: round(b[n].X, 1)             for n in DCs},
        'shortage': {
            s: round(sum(u[d, s, t].X for d in DEMANDS for t in periods), 2)
            for s in SCENARIOS
        },
    }
    r['stage1'] = sum(
        params['fixed_lane_cost'][l] * r['y'][l]
        for l in supply_lanes
    )
    r['stage2'] = r['obj'] - r['stage1']

    r['fuel_s'] = {
        s: sum(
            params['base_cost'][l] * params['cost_mult'][s]
            * x[l[0], l[1], s, t].X
            for l in ALL_LANES for t in periods
        )
        for s in SCENARIOS
    }
    r['shortage_s'] = {
        s: params['M'] * r['shortage'][s]
        for s in SCENARIOS
    }
    return r


def show(r, label=''):
    w = 56
    print(f'\n{"="*w}\n  {label}\n{"="*w}')
    print(f'  Expected total cost : {r["obj"]:,.0f}')
    print(f'  Stage 1 fixed       : {r["stage1"]:,.0f}')
    print(f'  Stage 2 expected    : {r["stage2"]:,.0f}')
    print('\n  Lane activation (Stage 1 binary decisions):')
    for l in supply_lanes:
        tag = 'ACTIVE' if r['y'][l] else 'CLOSED  ← model closed this'
        print(f'    {l[1]:16s}: {tag}')
    print('\n  Buffer stock pre-positioned:')
    for n, val in r['b'].items():
        print(f'    {n:16s}: {val:.1f} units')
    print('\n  Shortage by scenario:')
    for s in SCENARIOS:
        flag = ' ← demand deferred' if r['shortage'][s] > 0 else ' ← fully served'
        print(f'    {s:12s}: {r["shortage"][s]:.1f} units{flag}')


# ── Diagrams ──────────────────────────────────────────────────────────────────

def diagram_demand(params):
    dm  = params['demand_mult']
    bd  = params['base_demand']
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
    fig.suptitle(
        'Demand by node and scenario — Hormuz shock impact\n'
        'Red dashed line = lane capacity ceiling (60 units)',
        fontsize=11, fontweight='bold'
    )
    for ax, s in zip(axes, SCENARIOS):
        d1 = [bd[d] * dm[s]        for d in DEMANDS]
        d2 = [bd[d] * dm[s] * 1.1  for d in DEMANDS]
        xs = np.arange(len(DEMANDS))
        ax.bar(xs - 0.2, d1, 0.35, color=C[s], alpha=0.7, label='Period 1', ec='white')
        ax.bar(xs + 0.2, d2, 0.35, color=C[s], alpha=1.0, label='Period 2', ec='white')
        ax.axhline(60, color='#E24B4A', lw=1.2, ls='--', label='capacity=60')
        ax.set_xticks(xs)
        ax.set_xticklabels(DEMANDS)
        ax.set_title(
            f'{s.capitalize()}\nfuel ×{params["cost_mult"][s]:.2f}  '
            f'demand ×{dm[s]}',
            fontweight='bold', fontsize=10
        )
        ax.set_xlabel('Demand node')
        ax.set_ylabel('Units demanded')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig('diag1_demand.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('  Saved: diag1_demand.png')


def diagram_flows(result, params):
    mdl, y, b, x, u, v = result
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
    fig.suptitle(
        'Optimal flow on DC→demand lanes (Period 1) per scenario\n'
        'Dark border = at or near capacity ceiling',
        fontsize=11, fontweight='bold'
    )
    labels = [f'{l[0].replace("DC_","")[:3]}→{l[1]}' for l in dc_demand_lanes]
    for ax, s in zip(axes, SCENARIOS):
        flows = [x[l[0], l[1], s, 1].X for l in dc_demand_lanes]
        bars  = ax.barh(labels, flows, color=C[s], alpha=0.85, ec='white')
        ax.axvline(60, color='#E24B4A', lw=1, ls='--')
        for bar, flow in zip(bars, flows):
            if flow > 52:
                bar.set_edgecolor('#A32D2D')
                bar.set_linewidth(1.2)
        total_s = sum(u[d, s, t].X for d in DEMANDS for t in params['periods'])
        tc = '#A32D2D' if total_s > 0 else '#0F6E56'
        ax.set_title(
            f'{s.capitalize()}\nshortage: {total_s:.0f} units',
            fontweight='bold', fontsize=10, color=tc
        )
        ax.set_xlabel('Flow (units, period 1)')
    plt.tight_layout()
    plt.savefig('diag2_flows.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('  Saved: diag2_flows.png')


def diagram_costs(r_stoch, r_det):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        'Cost breakdown — stochastic vs deterministic\n'
        'CAVEAT: savings are relative to moderate-only baseline',
        fontsize=10, fontweight='bold'
    )

    ax1 = axes[0]
    xlabels = ['Mild\np=0.25', 'Moderate\np=0.50', 'Severe\np=0.25', 'Deterministic\n(mod only)']
    s1 = [r_stoch['stage1']] * 3 + [r_det['stage1']]
    fc = [r_stoch['fuel_s'][s]     for s in SCENARIOS] + [r_det['fuel_s']['moderate']]
    sc = [r_stoch['shortage_s'][s] for s in SCENARIOS] + [r_det['shortage_s']['moderate']]
    xs = np.arange(4)
    ax1.bar(xs, s1, color=C['stage1'], label='Stage 1 fixed', ec='white')
    ax1.bar(xs, fc, bottom=s1, color=C['moderate'], label='Stage 2 fuel', ec='white')
    b2 = [a + b for a, b in zip(s1, fc)]
    ax1.bar(xs, sc, bottom=b2, color=C['shortage'], label='Stage 2 shortage', ec='white')
    ax1.set_xticks(xs)
    ax1.set_xticklabels(xlabels, fontsize=9)
    ax1.set_ylabel('Cost')
    ax1.legend(fontsize=9)
    ax1.set_title('Per-scenario cost breakdown', fontweight='bold')

    ax2 = axes[1]
    stoch = r_stoch['obj']
    det   = r_det['obj']
    pct   = (det - stoch) / det * 100
    ax2.bar(
        ['Stochastic\noptimal', 'Deterministic\n(mod only)'],
        [stoch, det],
        color=[C['mild'], C['severe']],
        alpha=0.85, width=0.5, ec='white'
    )
    ax2.set_ylabel('Expected total cost')
    ax2.set_title('Value of stochastic solution\nCAVEAT: in-sample only', fontweight='bold', fontsize=9)
    ax2.annotate(
        f'VSS ≈ {det-stoch:,.0f}\n({pct:.1f}% savings)',
        xy=(0.5, (stoch + det) / 2),
        fontsize=10, ha='center', va='center',
        fontweight='bold', color='#0F6E56'
    )
    plt.tight_layout()
    plt.savefig('diag3_costs.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('  Saved: diag3_costs.png')


# ── What-if analysis ──────────────────────────────────────────────────────────

def run_whatifs(base_r):
    """
    Run 12 what-if analyses. Each changes ONE assumption and re-solves.
    CAVEAT: real disruptions change many assumptions at once.
    """
    print('\n' + '='*56)
    print('  WHAT-IF ANALYSIS — 12 configurations')
    print('  One assumption changed per run.')
    print('='*56)

    results = {'Base case': base_r['obj']}

    configs = [
        ('Severe more likely  p=0.60',
         {'prob': {'mild': 0.10, 'moderate': 0.30, 'severe': 0.60}}),

        ('Severe unlikely     p=0.10',
         {'prob': {'mild': 0.40, 'moderate': 0.50, 'severe': 0.10}}),

        ('Penalty M=20  (lenient)',
         {'M': 20.0}),

        ('Penalty M=100 (moderate)',
         {'M': 100.0}),

        ('Penalty M=200 (strict)',
         {'M': 200.0}),

        ('Demand shock ×5 severe',
         {'demand_mult': {'mild': 1.20, 'moderate': 1.80, 'severe': 5.00}}),

        ('Cost mult severe=2.0',
         {'cost_mult': {'mild': 1.10, 'moderate': 1.30, 'severe': 2.00}}),

        ('All lanes precontracted',
         {'fixed_lane_cost': {
             **{l: int(BASE_PARAMS['fixed_lane_cost'][l] * 0.5)
                for l in supply_lanes},
             **{l: 200 for l in dc_demand_lanes},
         }}),

        ('India: low reserve',
         {'init_inv':    {'DC_North': 12, 'DC_Central': 15, 'DC_South': 10},
          'demand_mult': {'mild': 1.30, 'moderate': 2.20, 'severe': 4.50},
          'M': 120.0}),

        ('Japan: deep reserve',
         {'init_inv':    {'DC_North': 200, 'DC_Central': 220, 'DC_South': 180},
          'demand_mult': {'mild': 1.10, 'moderate': 1.40, 'severe': 2.00},
          'M': 60.0}),

        ('Singapore: hub repricing',
         {'demand_mult': {'mild': 1.50, 'moderate': 2.50, 'severe': 5.00},
          'cost_mult':   {'mild': 1.15, 'moderate': 1.45, 'severe': 1.80},
          'M': 90.0}),

        ('T=3 periods extended',
         {'periods': [1, 2, 3]}),
    ]

    for label, overrides in configs:
        p = deepcopy(BASE_PARAMS)
        for k, v in overrides.items():
            if isinstance(v, dict) and isinstance(p.get(k), dict):
                p[k] = {**p[k], **v}
            else:
                p[k] = v

        res = build_and_solve(p, silent=True)
        if res:
            rr = extract(res, p)
            delta = rr['obj'] - base_r['obj']
            changed = [
                l[1] for l in supply_lanes
                if rr['y'][l] != base_r['y'][l]
            ]
            print(f'\n  [{label}]')
            print(f'    obj = {rr["obj"]:,.0f}   '
                  f'delta = {delta:+,.0f}   '
                  f'lane changes: {changed if changed else "none"}')
            results[label] = rr['obj']
        else:
            print(f'\n  [{label}] — solve failed')

    # ── Sensitivity plot ──────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(13, 7))
    labels = list(results.keys())
    vals   = list(results.values())
    base   = results['Base case']
    colors = [
        '#5DCAA5' if v <= base * 1.02 else
        '#EF9F27' if v <= base * 1.20 else
        '#D85A30'
        for v in vals
    ]
    xs = np.arange(len(labels))
    ax.barh(xs, vals, color=colors, alpha=0.85, ec='white', height=0.6)
    ax.axvline(base, color='#888780', lw=1.2, ls='--',
               label=f'Base: {base:,.0f}')
    ax.set_yticks(xs)
    ax.set_yticklabels(labels, fontsize=9.5)
    ax.set_xlabel('Expected total cost')
    ax.set_title(
        'What-if sensitivity — 12 assumption changes\n'
        'CAVEAT: each bar changes ONE parameter. Real shocks change many.',
        fontweight='bold', fontsize=10
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('diag4_whatif.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\n  Saved: diag4_whatif.png')

    return results


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    print('='*56)
    print('  HORMUZ OPTIMISATION PROBLEM')
    print('  Two-Stage Stochastic MILP  |  Gurobi')
    print()
    print('  REMINDER: This is a conjecture, not a prescription.')
    print('  Every parameter is an assumption.')
    print('='*56)

    # ── Base case ──────────────────────────────────────────────────────────
    print('\nSolving base case...')
    res_base = build_and_solve(BASE_PARAMS, silent=False)
    if not res_base:
        print('Base case failed. Check Gurobi licence.')
        return

    r_base = extract(res_base, BASE_PARAMS)
    show(r_base, 'BASE CASE — stochastic optimal')

    # ── Deterministic baseline (moderate only) ─────────────────────────────
    print('\nSolving deterministic baseline (moderate scenario only)...')
    p_det = deepcopy(BASE_PARAMS)
    p_det['prob'] = {'mild': 0.0, 'moderate': 1.0, 'severe': 0.0}
    res_det = build_and_solve(p_det, silent=True)
    r_det = extract(res_det, p_det) if res_det else r_base

    vss = r_det['obj'] - r_base['obj']
    print(f'\n  Value of Stochastic Solution (VSS): {vss:,.0f}')
    print(f'  ({vss / r_det["obj"] * 100:.1f}% savings vs moderate-only plan)')
    print(f'  CAVEAT: in-sample VSS — out-of-sample may differ.')

    # ── Diagrams ───────────────────────────────────────────────────────────
    print('\nGenerating diagrams...')
    diagram_demand(BASE_PARAMS)
    diagram_flows(res_base, BASE_PARAMS)
    diagram_costs(r_base, r_det)

    # ── What-if ────────────────────────────────────────────────────────────
    run_whatifs(r_base)

    print('\n' + '='*56)
    print('  LIMITATIONS SUMMARY')
    print('  1. Scenario probabilities are assumptions, not statistics')
    print('  2. Demand multipliers estimate emergent panic behaviour')
    print('  3. Contract penalties, regulation, correlation not modelled')
    print('  4. Production scale (50+ DCs) needs Benders decomposition')
    print('  5. Use as decision support, not as a decision')
    print('='*56)


if __name__ == '__main__':
    main()

  HORMUZ OPTIMISATION PROBLEM
  Two-Stage Stochastic MILP  |  Gurobi

  REMINDER: This is a conjecture, not a prescription.
  Every parameter is an assumption.

Solving base case...
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.001
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Non-default parameters:
MIPGap  0.001

Optimize a model with 153 rows, 120 columns and 312 nonzeros (Min)
Model fingerprint: 0x512303f1
Model has 99 linear objective coefficients
Variable types: 117 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [1e-01, 9e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+01, 1e+02]

Presolve removed 93 rows and 18 columns
Presolve time: 0.01s
Presolved: 60 rows, 102 columns, 184 nonzeros
Variable ty